# Deutsche Bahn Historical Delay Data Catalog

## 1. Purpose, scope, and evidence rules

This notebook creates a reproducible **dataset-level and column-level data catalog**
for the monthly Deutsche Bahn historical delay Parquet files used by the capstone project.

It documents:

- source, coverage, refresh cadence, and technical format;
- the proposed row grain and identifier roles;
- schema compatibility across July 2024 and July 2026;
- sample row counts and temporal coverage;
- column meanings, formats, derivations, and known quality risks;
- evidence-backed validation results and unresolved questions.

The catalog distinguishes four confidence levels:

| Status | Meaning |
|---|---|
| `observed` | Calculated directly from the loaded Parquet samples |
| `source_documented` | Taken from the source repository or processing logic |
| `hypothesis` | Plausible interpretation that still requires validation |
| `to_verify` | Important metadata that is not yet sufficiently evidenced |

Detailed null percentages, distributions, and anomaly analysis are intentionally
deferred to the profiling and data-quality notebooks.

In [33]:
from datetime import datetime, timezone
from html import escape
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

from pyspark.sql import DataFrame, SparkSession, functions as F


DATA_DIRECTORY = Path(
    "/opt/spark/work-dir/data/historical_delays"
)

DATA_FILES = [
    "data-2024-07.parquet",
    "data-2026-07.parquet",
]

CATALOG_BASE_PATH = Path(
    "/opt/spark/work-dir/artifacts/catalog/deutsche_bahn_historical"
)

CATALOG_GENERATED_AT_UTC = datetime.now(
    timezone.utc
).isoformat()

for data_file in DATA_FILES:
    data_path = DATA_DIRECTORY / data_file

    print("=" * 60)
    print("Dataset:", data_file)
    print("Dataset path:", data_path)
    print("Dataset exists:", data_path.exists())

    if data_path.exists():
        print(
            "Dataset size (MB):",
            round(data_path.stat().st_size / 1024**2, 2),
        )
    else:
        print("Dataset file was not found.")

Dataset: data-2024-07.parquet
Dataset path: /opt/spark/work-dir/data/historical_delays/data-2024-07.parquet
Dataset exists: True
Dataset size (MB): 101.87
Dataset: data-2026-07.parquet
Dataset path: /opt/spark/work-dir/data/historical_delays/data-2026-07.parquet
Dataset exists: True
Dataset size (MB): 563.05


## 2. Reuse or create the Spark session

The Parquet source stores timestamp fields with **nanosecond precision**.
Spark 3.5 cannot directly interpret `TIMESTAMP(NANOS, false)` as a normal
Spark timestamp.

The compatibility option below exposes these fields as `long` values. The
notebook then converts the known timestamp columns from nanoseconds to
microseconds before creating Spark timestamps.

The Spark session uses UTC for deterministic technical decoding. The business
timezone is interpreted as `Europe/Berlin`, but the catalog marks that
interpretation as requiring explicit source confirmation because the physical
Parquet type is timezone-naive.

In [2]:
SOURCE_TIME_ZONE = "Europe/Berlin"

try:
    spark
except NameError:
    spark = (
        SparkSession.builder
        .appName("deutsche-bahn-data-catalog")
        .master("spark://spark-master:7077")
        .config("spark.driver.host", "spark-jupyter")
        .config("spark.driver.bindAddress", "0.0.0.0")
        .getOrCreate()
    )

spark.sparkContext.setLogLevel("WARN")

# Technical timezone used while decoding the raw epoch-based integers.
spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC",
)

# Read unsupported Parquet nanosecond timestamps as long integers.
spark.conf.set(
    "spark.sql.legacy.parquet.nanosAsLong",
    "true",
)

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)
print(
    "Technical decoding timezone:",
    spark.conf.get("spark.sql.session.timeZone"),
)
print(
    "Source timestamp timezone:",
    SOURCE_TIME_ZONE,
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/05 09:43:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.8
Spark master: spark://spark-master:7077
Technical decoding timezone: UTC
Source timestamp timezone: Europe/Berlin


## 3. Load the monthly Parquet validation samples

- `historical_df_072024_raw`: July 2024 source schema exactly as Spark reads it;
- `historical_df_072026_raw`: July 2026 source schema exactly as Spark reads it.

The `_raw` suffix is important: timestamp fields remain physical `long`
nanosecond values until the explicit conversion step.

In [3]:

DATA_PATH_072024 = (
    DATA_DIRECTORY / "data-2024-07.parquet"
)

DATA_PATH_072026 = (
    DATA_DIRECTORY / "data-2026-07.parquet"
)


required_paths = {
    "2024-07": DATA_PATH_072024,
    "2026-07": DATA_PATH_072026,
}

for period, data_path in required_paths.items():
    if not data_path.exists():
        raise FileNotFoundError(
            f"Dataset for {period} not found at {data_path}. "
            "Download the monthly Parquet file first."
        )

    print(
        f"{period}: {data_path.name} "
        f"({data_path.stat().st_size / 1024**2:.2f} MB)"
    )

2024-07: data-2024-07.parquet (101.87 MB)
2026-07: data-2026-07.parquet (563.05 MB)


In [4]:
historical_df_072024_raw = spark.read.parquet(
    str(DATA_PATH_072024)
)

historical_df_072026_raw = spark.read.parquet(
    str(DATA_PATH_072026)
)

raw_datasets = {
    "2024-07": historical_df_072024_raw,
    "2026-07": historical_df_072026_raw,
}

In [5]:
print("=" * 80)
print("Raw schema — July 2024")
print("=" * 80)

historical_df_072024_raw.printSchema()


print("=" * 80)
print("Raw schema — July 2026")
print("=" * 80)

historical_df_072026_raw.printSchema()

Raw schema — July 2024
root
 |-- station_name: string (nullable = true)
 |-- xml_station_name: string (nullable = true)
 |-- eva: string (nullable = true)
 |-- train_number: string (nullable = true)
 |-- line_number: string (nullable = true)
 |-- final_destination_station: string (nullable = true)
 |-- delay_in_min: integer (nullable = true)
 |-- time: long (nullable = true)
 |-- is_canceled: boolean (nullable = true)
 |-- train_type: string (nullable = true)
 |-- train_line_ride_id: string (nullable = true)
 |-- train_line_station_num: integer (nullable = true)
 |-- arrival_planned_time: long (nullable = true)
 |-- arrival_change_time: long (nullable = true)
 |-- departure_planned_time: long (nullable = true)
 |-- departure_change_time: long (nullable = true)
 |-- id: string (nullable = true)

Raw schema — July 2026
root
 |-- station_name: string (nullable = true)
 |-- xml_station_name: string (nullable = true)
 |-- eva: string (nullable = true)
 |-- train_number: string (nullable = t

### Check whether the schemas are exactly identical

The comparison is calculated from the loaded DataFrames. 

In [6]:
schemas_are_identical = (
    historical_df_072024_raw.schema
    == historical_df_072026_raw.schema
)

print(
    "Schemas are identical:",
    schemas_are_identical,
)

Schemas are identical: True


### Compare columns and data types

In [7]:
schema_072024 = {
    field.name: {
        "data_type": field.dataType.simpleString(),
        "nullable": field.nullable,
    }
    for field in historical_df_072024_raw.schema.fields
}

schema_072026 = {
    field.name: {
        "data_type": field.dataType.simpleString(),
        "nullable": field.nullable,
    }
    for field in historical_df_072026_raw.schema.fields
}

In [8]:
columns_072024 = set(schema_072024)
columns_072026 = set(schema_072026)

only_in_072024 = sorted(
    columns_072024 - columns_072026
)

only_in_072026 = sorted(
    columns_072026 - columns_072024
)

print(
    "Columns only in July 2024:",
    only_in_072024,
)

print(
    "Columns only in July 2026:",
    only_in_072026,
)

Columns only in July 2024: []
Columns only in July 2026: []


In [9]:
common_columns = sorted(
    columns_072024 & columns_072026
)

schema_differences = []
schema_comparison_rows = []

for column_name in sorted(
    columns_072024 | columns_072026
):
    definition_072024 = schema_072024.get(column_name)
    definition_072026 = schema_072026.get(column_name)

    definitions_match = (
        definition_072024 == definition_072026
    )

    schema_comparison_rows.append(
        (
            column_name,
            (
                definition_072024["data_type"]
                if definition_072024
                else None
            ),
            (
                definition_072026["data_type"]
                if definition_072026
                else None
            ),
            (
                definition_072024["nullable"]
                if definition_072024
                else None
            ),
            (
                definition_072026["nullable"]
                if definition_072026
                else None
            ),
            definitions_match,
        )
    )

    if not definitions_match:
        schema_differences.append(
            {
                "column_name": column_name,
                "definition_2024_07": definition_072024,
                "definition_2026_07": definition_072026,
            }
        )

schema_comparison_df = spark.createDataFrame(
    schema_comparison_rows,
    (
        "column_name string, "
        "type_2024_07 string, "
        "type_2026_07 string, "
        "nullable_2024_07 boolean, "
        "nullable_2026_07 boolean, "
        "definitions_match boolean"
    ),
)

print(
    "Number of schema differences:",
    len(schema_differences),
)

for difference in schema_differences:
    print(difference)

Number of schema differences: 0


In [10]:
print("=" * 80)
print("Raw sample — July 2024")
print("=" * 80)

historical_df_072024_raw.show(
    1,
    truncate=False,
    vertical=True,
)


print("=" * 80)
print("Raw sample — July 2026")
print("=" * 80)

historical_df_072026_raw.show(
    1,
    truncate=False,
    vertical=True,
)

Raw sample — July 2024
-RECORD 0------------------------------------------------------
 station_name              | NULL                              
 xml_station_name          | ZOB/Hauptbahnhof, Pforzheim       
 eva                       | 0940370                           
 train_number              | 33382                             
 line_number               | S6 (S                             
 final_destination_station | Bahnhof, Bad Wildbad              
 delay_in_min              | 0                                 
 time                      | 1719792000000000000               
 is_canceled               | false                             
 train_type                | Bus                               
 train_line_ride_id        | -6129702905591104469              
 train_line_station_num    | 1                                 
 arrival_planned_time      | NULL                              
 arrival_change_time       | NULL                              
 departure_planne

## 4. Define and test the row-grain hypothesis

### Current grain hypothesis

> One row appears to represent one processed service-stop observation for one
> train ride at one station sequence position.

The identifiers have different roles:

- `id` is the candidate **row-level primary key**;
- `train_line_ride_id` is a **parent ride identifier** and is expected to
  repeat across the stations served by that ride;
- `train_line_ride_id + train_line_station_num` is a candidate
  **business key** for a stop within a ride.

This corrects the earlier assumption that `train_line_ride_id` alone could be a
natural row key. A single ride normally produces multiple station observations,
so duplicate ride identifiers are expected rather than automatically erroneous.

In [11]:
TIMESTAMP_COLUMNS = [
    "time",
    "arrival_planned_time",
    "arrival_change_time",
    "departure_planned_time",
    "departure_change_time",
]


def convert_nanosecond_timestamps(
    dataframe: DataFrame,
    timestamp_columns: list[str],
) -> DataFrame:
    # Convert raw nanosecond integers to Spark timestamps.
    missing_columns = sorted(
        set(timestamp_columns) - set(dataframe.columns)
    )

    if missing_columns:
        raise ValueError(
            "Timestamp columns missing from the schema: "
            f"{missing_columns}"
        )

    transformed = dataframe

    for column_name in timestamp_columns:
        transformed = transformed.withColumn(
            column_name,
            F.when(
                F.col(column_name).isNotNull(),
                F.timestamp_micros(
                    F.expr(
                        f"`{column_name}` DIV 1000"
                    )
                ),
            ).otherwise(
                F.lit(None).cast("timestamp")
            ),
        )

    return transformed

In [12]:
historical_df_072024 = (
    convert_nanosecond_timestamps(
        historical_df_072024_raw,
        TIMESTAMP_COLUMNS,
    )
)

historical_df_072026 = (
    convert_nanosecond_timestamps(
        historical_df_072026_raw,
        TIMESTAMP_COLUMNS,
    )
)

decoded_datasets = {
    "2024-07": historical_df_072024,
    "2026-07": historical_df_072026,
}

In [13]:
#Create a reusable summary function
def summarize_dataset(
    dataframe: DataFrame,
    period: str,
    file_path: Path,
) -> dict:
    """
    Calculate basic catalog metadata for one monthly file.
    """

    summary = (
        dataframe
        .agg(
            F.count("*").alias("row_count"),
            F.min("time").alias("minimum_time"),
            F.max("time").alias("maximum_time"),
            F.countDistinct(
                F.to_date("time")
            ).alias("distinct_service_dates"),
        )
        .first()
        .asDict()
    )

    return {
        "period": period,
        "file_name": file_path.name,
        "file_path": str(file_path),
        "file_size_mb": round(
            file_path.stat().st_size / 1024**2,
            2,
        ),
        "row_count": summary["row_count"],
        "column_count": len(dataframe.columns),
        "minimum_time": summary["minimum_time"],
        "maximum_time": summary["maximum_time"],
        "distinct_service_dates": (
            summary["distinct_service_dates"]
        ),
    }

In [14]:
summary_072024 = summarize_dataset(
    dataframe=historical_df_072024,
    period="2024-07",
    file_path=DATA_PATH_072024,
)

summary_072026 = summarize_dataset(
    dataframe=historical_df_072026,
    period="2026-07",
    file_path=DATA_PATH_072026,
)

dataset_summaries = {
    "2024-07": summary_072024,
    "2026-07": summary_072026,
}

sample_summary_rows = [
    (
        summary["period"],
        summary["file_name"],
        summary["file_path"],
        summary["file_size_mb"],
        summary["row_count"],
        summary["column_count"],
        summary["minimum_time"],
        summary["maximum_time"],
        summary["distinct_service_dates"],
    )
    for summary in dataset_summaries.values()
]

sample_summary_df = spark.createDataFrame(
    sample_summary_rows,
    (
        "period string, "
        "file_name string, "
        "file_path string, "
        "file_size_mb double, "
        "row_count long, "
        "column_count int, "
        "minimum_time timestamp, "
        "maximum_time timestamp, "
        "distinct_service_dates long"
    ),
)

In [15]:
for period, summary in dataset_summaries.items():
    print("=" * 70)
    print("Period:", period)
    print("File:", summary["file_name"])
    print(f"File size: {summary['file_size_mb']} MB")
    print(f"Rows: {summary['row_count']:,}")
    print(f"Columns: {summary['column_count']}")
    print("Minimum time:", summary["minimum_time"])
    print("Maximum time:", summary["maximum_time"])
    print(
        "Distinct service dates:",
        summary["distinct_service_dates"],
    )

Period: 2024-07
File: data-2024-07.parquet
File size: 101.87 MB
Rows: 2,007,251
Columns: 17
Minimum time: 2024-07-01 00:00:00
Maximum time: 2024-07-31 23:59:00
Distinct service dates: 31
Period: 2026-07
File: data-2026-07.parquet
File size: 563.05 MB
Rows: 14,052,153
Columns: 17
Minimum time: 2026-07-01 00:00:00
Maximum time: 2026-07-31 23:59:00
Distinct service dates: 31


## 5. Test candidate identifiers in both validation months

A candidate key must be checked for both nulls and duplicates. The assessment
is run against July 2024 and July 2026 so that a result is not based only on
the latest, much larger file.

In [20]:
%pip install pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 11.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.3/14.3 MB 12.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.3/508.3 KB 11.7 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [23]:
def assess_identifier(
    dataframe: DataFrame,
    period: str,
    key_name: str,
    key_type: str,
    key_columns: list[str],
    expected_unique: bool,
) -> tuple:
    null_condition = F.lit(False)

    for column_name in key_columns:
        null_condition = (
            null_condition
            | F.col(column_name).isNull()
        )

    null_key_row_count = (
        dataframe
        .filter(null_condition)
        .count()
    )

    duplicate_group_count = (
        dataframe
        .groupBy(*key_columns)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    passes_initial_check = (
        null_key_row_count == 0
        and duplicate_group_count == 0
        if expected_unique
        else None
    )

    return (
        period,
        key_name,
        key_type,
        json.dumps(key_columns),
        expected_unique,
        null_key_row_count,
        duplicate_group_count,
        passes_initial_check,
    )


identifier_definitions = [
    {
        "key_name": "id",
        "key_type": "candidate_primary_key",
        "key_columns": ["id"],
        "expected_unique": True,
    },
    {
        "key_name": "train_line_ride_id",
        "key_type": "parent_ride_identifier",
        "key_columns": ["train_line_ride_id"],
        "expected_unique": False,
    },
    {
        "key_name": (
            "train_line_ride_id + "
            "train_line_station_num"
        ),
        "key_type": "candidate_business_key",
        "key_columns": [
            "train_line_ride_id",
            "train_line_station_num",
        ],
        "expected_unique": True,
    },
]

key_assessment_rows = []

for period, dataframe in raw_datasets.items():
    for definition in identifier_definitions:
        key_assessment_rows.append(
            assess_identifier(
                dataframe=dataframe,
                period=period,
                **definition,
            )
        )

key_assessment_df = spark.createDataFrame(
    key_assessment_rows,
    (
        "period string, "
        "key_name string, "
        "key_type string, "
        "key_columns_json string, "
        "expected_unique boolean, "
        "null_key_row_count long, "
        "duplicate_group_count long, "
        "passes_initial_check boolean"
    ),
)

key_assessment_pd = (
    key_assessment_df
    .orderBy(
        "period",
        "key_type",
    )
    .toPandas()
)

key_type_labels = {
    "candidate_primary_key": "Candidate primary key",
    "parent_ride_identifier": "Parent ride identifier",
    "candidate_business_key": "Candidate business key",
}

key_type_order = [
    "candidate_primary_key",
    "parent_ride_identifier",
    "candidate_business_key",
]

key_assessment_pd["key_type"] = pd.Categorical(
    key_assessment_pd["key_type"],
    categories=key_type_order,
    ordered=True,
)

key_assessment_pd = (
    key_assessment_pd
    .sort_values(
        ["period", "key_type"],
        ignore_index=True,
    )
)

key_assessment_display_df = pd.DataFrame(
    {
        "Period": key_assessment_pd["period"],
        "Identifier": key_assessment_pd["key_name"],
        "Role": (
            key_assessment_pd["key_type"]
            .astype("object")
            .map(key_type_labels)
        ),
        "Columns": (
            key_assessment_pd["key_columns_json"]
            .map(
                lambda value: " + ".join(
                    json.loads(value)
                )
            )
        ),
        "Expected unique": (
            key_assessment_pd["expected_unique"]
            .map(
                {
                    True: "Yes",
                    False: "No",
                }
            )
        ),
        "Null key rows": (
            key_assessment_pd["null_key_row_count"]
            .astype("int64")
        ),
        "Duplicate groups": (
            key_assessment_pd["duplicate_group_count"]
            .astype("int64")
        ),
        "Assessment": (
            key_assessment_pd["passes_initial_check"]
            .map(
                {
                    True: "Pass",
                    False: "Fail",
                }
            )
            .fillna("Informational")
        ),
    }
)


def style_assessment_status(
    column: pd.Series,
) -> list[str]:
    styles = {
        "Pass": (
            "background-color: #dcfce7; "
            "color: #166534; "
            "font-weight: 700;"
        ),
        "Fail": (
            "background-color: #fee2e2; "
            "color: #991b1b; "
            "font-weight: 700;"
        ),
        "Informational": (
            "background-color: #e0f2fe; "
            "color: #075985; "
            "font-weight: 700;"
        ),
    }

    return [
        styles.get(value, "")
        for value in column
    ]


key_assessment_styled = (
    key_assessment_display_df
    .style
    .hide(axis="index")
    .set_caption(
        "Identifier assessment across monthly validation samples"
    )
    .format(
        {
            "Null key rows": "{:,}",
            "Duplicate groups": "{:,}",
        }
    )
    .set_properties(
        **{
            "text-align": "left",
            "vertical-align": "top",
            "white-space": "normal",
        }
    )
    .set_properties(
        subset=[
            "Period",
            "Expected unique",
            "Null key rows",
            "Duplicate groups",
            "Assessment",
        ],
        **{
            "text-align": "center",
            "white-space": "nowrap",
        },
    )
    .set_properties(
        subset=[
            "Identifier",
            "Columns",
        ],
        **{
            "font-family": "monospace",
        },
    )
    .set_table_styles(
        [
            {
                "selector": "caption",
                "props": [
                    ("caption-side", "top"),
                    ("font-size", "16px"),
                    ("font-weight", "700"),
                    ("text-align", "left"),
                    ("padding", "0 0 10px 0"),
                ],
            },
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("width", "100%"),
                    ("font-size", "13px"),
                ],
            },
            {
                "selector": "th",
                "props": [
                    ("background-color", "#f3f4f6"),
                    ("color", "#111827"),
                    ("font-weight", "700"),
                    ("border", "1px solid #d1d5db"),
                    ("padding", "8px"),
                    ("text-align", "left"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("border", "1px solid #e5e7eb"),
                    ("padding", "8px"),
                ],
            },
            {
                "selector": "tbody tr:nth-child(even)",
                "props": [
                    ("background-color", "#f9fafb"),
                ],
            },
            {
                "selector": "tbody tr:hover",
                "props": [
                    ("background-color", "#f3f4f6"),
                ],
            },
        ]
    )
    .apply(
        style_assessment_status,
        subset=["Assessment"],
    )
)

display(key_assessment_styled)

id_passes_all_samples = (
    key_assessment_df
    .filter(F.col("key_name") == "id")
    .filter(F.col("passes_initial_check") != True)
    .count()
    == 0
)

business_key_passes_all_samples = (
    key_assessment_df
    .filter(
        F.col("key_type")
        == "candidate_business_key"
    )
    .filter(F.col("passes_initial_check") != True)
    .count()
    == 0
)

Period,Identifier,Role,Columns,Expected unique,Null key rows,Duplicate groups,Assessment
2024-07,id,Candidate primary key,id,Yes,0,0,Pass
2024-07,train_line_ride_id,Parent ride identifier,train_line_ride_id,No,0,"57,038",Informational
2024-07,train_line_ride_id + train_line_station_num,Candidate business key,train_line_ride_id + train_line_station_num,Yes,0,"119,304",Fail
2026-07,id,Candidate primary key,id,Yes,0,0,Pass
2026-07,train_line_ride_id,Parent ride identifier,train_line_ride_id,No,0,"85,478",Informational
2026-07,train_line_ride_id + train_line_station_num,Candidate business key,train_line_ride_id + train_line_station_num,Yes,0,"863,139",Fail


## 6. Define the dataset-level catalog

The dataset catalog combines source metadata, runtime observations, hypotheses,
and open verification items. Each exported attribute includes a
`verification_status` and an `evidence` field so that observed facts are not
mixed with undocumented assumptions.

In [47]:
dataset_catalog = {
    "dataset_name": (
        "deutsche_bahn_historical_delays"
    ),
    "display_name": (
        "Deutsche Bahn Historical Delay Data"
    ),

    # Source
    "geographic_scope": "Germany",
    "source_platform": "Hugging Face",
    "source_repository": (
        "piebro/deutsche-bahn-data"
    ),
    "source_url": (
        "https://huggingface.co/datasets/"
        "piebro/deutsche-bahn-data"
    ),

    # Availability and coverage
    "available_data_period": (
        "July 2024 onward"
    ),
    "coverage_transition_date": (
        "2025-11-02"
    ),
    "coverage_before_transition": (
        "From July 2024 through 2025-11-01: "
        "data for approximately the 100 largest "
        "railway stations"
    ),
    "coverage_from_transition": (
        "From 2025-11-02 onward: data for all "
        "available Deutsche Bahn stations"
    ),

    # Refresh frequency
    "raw_api_collection_frequency": (
        "Four times per day, approximately "
        "every six hours"
    ),
    "processed_release_frequency": (
        "Monthly — one processed Parquet file "
        "per month"
    ),

    # Technical format
    "source_format": "Parquet",
    "timestamp_storage": (
        "Nanoseconds"
    ),
    "source_operational_timezone": (
        "Europe/Berlin (CET/CEST)"
    ),

    # Validation samples
    "sample_files": [
        summary_072024["file_name"],
        summary_072026["file_name"],
    ],
    "sample_periods": [
        "2024-07",
        "2026-07",
    ],

    "sample_time_scopes": {
        "2024-07": (
            f"{summary_072024['minimum_time']} "
            f"to {summary_072024['maximum_time']}"
        ),
        "2026-07": (
            f"{summary_072026['minimum_time']} "
            f"to {summary_072026['maximum_time']}"
        ),
    },

    "sample_row_counts": {
        "2024-07": summary_072024["row_count"],
        "2026-07": summary_072026["row_count"],
    },

    "sample_distinct_service_dates": {
        "2024-07": (
            summary_072024[
                "distinct_service_dates"
            ]
        ),
        "2026-07": (
            summary_072026[
                "distinct_service_dates"
            ]
        ),
    },

    # Schema validation
    "schema_validation_samples": (
        "July 2024 and July 2026"
    ),
    "schemas_identical": True,
    "columns_only_in_2024_07": [],
    "columns_only_in_2026_07": [],
    "schema_difference_count": 0,
    "column_count": (
        summary_072024["column_count"]
    ),

    # Grain and keys
    "grain_hypothesis": (
        "One service observation for one ride "
        "at one station in the ride sequence"
    ),
    "candidate_primary_key": "id",
    "candidate_primary_key_passes_samples": "True",

    # Status
    "schema_status": (
        "Verified across July 2024 and July 2026"
    ),
    "catalog_status": (
        "Structurally complete for the two validation samples; "
        "full-history samples still require verification"
    ),
}



In [48]:
dataset_catalog_rows = []

for attribute, value in dataset_catalog.items():
    if isinstance(value, (dict, list)):
        catalog_value = json.dumps(
            value,
            ensure_ascii=False,
            default=str,
        )
    else:
        catalog_value = str(value)

    dataset_catalog_rows.append(
        (
            attribute,
            catalog_value,
        )
    )

dataset_catalog_df = spark.createDataFrame(
    dataset_catalog_rows,
    [
        "catalog_attribute",
        "catalog_value",
    ],
)

dataset_catalog_df.show(
    n=len(dataset_catalog_rows),
    truncate=False,
)

+------------------------------------+------------------------------------------------------------------------------------------------------------------+
|catalog_attribute                   |catalog_value                                                                                                     |
+------------------------------------+------------------------------------------------------------------------------------------------------------------+
|dataset_name                        |deutsche_bahn_historical_delays                                                                                   |
|display_name                        |Deutsche Bahn Historical Delay Data                                                                               |
|geographic_scope                    |Germany                                                                                                           |
|source_platform                     |Hugging Face                          

## 7. Define the business column catalog

Each physical field receives:

- a business description;
- a unit or representation;
- a semantic role;
- a source or derivation note;
- a data-quality note;
- a verification status.

> **Timestamp interpretation:** Raw timestamp fields are timezone-naive
> nanosecond values. They are decoded technically in UTC. `Europe/Berlin`
> remains the intended business interpretation, but it should not be presented
> as fully verified until confirmed from the upstream processing logic.

In [27]:
column_metadata = {
    "station_name": {
        "description": (
            "Station name resolved from the EVA-to-station-name "
            "mapping used during monthly processing. It may be null "
            "when the EVA number is not present in that mapping."
        ),
        "unit_or_format": "string, nullable",
        "verification_status": (
            "source_documented"
        ),
    },
    "xml_station_name": {
        "description": (
            "Station name taken from the root `station` attribute "
            "of the Deutsche Bahn Timetables XML response."
        ),
        "unit_or_format": "string",
        "verification_status": (
            "source_documented"
        ),
    },
    "eva": {
        "description": (
            "EVA station number: the Deutsche Bahn station "
            "identifier used in Timetables API requests."
        ),
        "unit_or_format": "string identifier",
        "verification_status": (
            "source_documented"
        ),
    },
    "train_number": {
        "description": (
            "Raw train number (`tl.n`, Zugnummer) identifying a "
            "specific train run, for example `123` or `12603`. "
            "Combine with `train_type` for labels such as `ICE 123`."
        ),
        "unit_or_format": "string",
        "verification_status": (
            "source_documented"
        ),
    },
    "line_number": {
        "description": (
            "Raw route or line number from `ar.l` or `dp.l`. It is "
            "non-unique across runs and is commonly null for "
            "long-distance trains such as ICE, IC, and EC."
        ),
        "unit_or_format": "string, nullable",
        "verification_status": (
            "source_documented"
        ),
    },
    "final_destination_station": {
        "description": (
            "Final destination of the train."
        ),
        "unit_or_format": "string",
        "verification_status": (
            "source_documented"
        ),
    },
    "delay_in_min": {
        "description": (
            "Delay in minutes."
        ),
        "unit_or_format": "integer minutes",
        "verification_status": (
            "source_documented"
        ),
    },
    "time": {
        "description": (
            "Actual arrival or departure time."
        ),
        "unit_or_format": (
            "raw long in nanoseconds; decoded as Spark timestamp, business timezone to verify"
        ),
        "verification_status": (
            "source_documented"
        ),
    },
    "is_canceled": {
        "description": (
            "Indicates whether the train stop was canceled. The value "
            "is derived from the presence of arrival or departure "
            "cancellation timestamps (`ar.clt` or `dp.clt`)."
        ),
        "unit_or_format": "boolean",
        "verification_status": (
            "source_documented"
        ),
    },
    "train_type": {
        "description": (
            "Raw train category from `tl.c`, for example ICE, IC, RE, "
            "RB, or S."
        ),
        "unit_or_format": "string",
        "verification_status": (
            "source_documented"
        ),
    },
    "train_line_ride_id": {
        "description": (
            "Identifier shared by station observations belonging to the same processed train ride."
        ),
        "unit_or_format": "string",
        "verification_status": "source_documented",
    },
    "train_line_station_num": {
        "description": (
            "Station position within the train ride, parsed from the "
            "last component of the source train-stop `id`."
        ),
        "unit_or_format": "integer",
        "verification_status": (
            "source_documented"
        ),
    },
    "arrival_planned_time": {
        "description": (
            "Planned arrival timestamp from the Timetables XML "
            "arrival attribute `ar.pt`."
        ),
        "unit_or_format": (
            "raw long in nanoseconds; decoded as Spark timestamp, business timezone to verify. "
        ),
        "verification_status": (
            "source_documented"
        ),
    },
    "arrival_change_time": {
        "description": (
            "Effective arrival timestamp. It uses the changed/actual "
            "arrival time from `ar.ct` when available and otherwise "
            "falls back to `arrival_planned_time`."
        ),
        "unit_or_format": (
            "raw long in nanoseconds; decoded as Spark timestamp, business timezone to verify."
        ),
        "verification_status": (
            "source_documented"
        ),
    },
    "departure_planned_time": {
        "description": (
            "Planned departure timestamp from the Timetables XML "
            "departure attribute `dp.pt`."
        ),
        "unit_or_format": (
            "raw long in nanoseconds; decoded as Spark timestamp, business timezone to verify."
        ),
        "verification_status": (
            "source_documented"
        ),
    },
    "departure_change_time": {
        "description": (
            "Effective departure timestamp. It uses the changed/actual "
            "departure time from `dp.ct` when available and otherwise "
            "falls back to `departure_planned_time`."
        ),
        "unit_or_format": (
            "raw long in nanoseconds; decoded as Spark timestamp, business timezone to verify."
        ),
        "verification_status": (
            "source_documented"
        ),
    },
    "id": {
        "description": (
            "Unique identifier for the train stop."
        ),
        "unit_or_format": "string",
        "verification_status": (
            "source_documented"
        ),
    },
}


column_roles = {
    "station_name": "dimension",
    "xml_station_name": "dimension",
    "eva": "identifier",
    "train_number": "identifier",
    "line_number": "dimension",
    "final_destination_station": "dimension",
    "delay_in_min": "measure",
    "time": "event_time",
    "is_canceled": "flag",
    "train_type": "dimension",
    "train_line_ride_id": "parent_identifier",
    "train_line_station_num": "sequence",
    "arrival_planned_time": "event_time",
    "arrival_change_time": "event_time",
    "departure_planned_time": "event_time",
    "departure_change_time": "event_time",
    "id": "row_identifier",
}

column_sources = {
    "station_name": "EVA-to-station-name lookup",
    "xml_station_name": "Timetables XML station attribute",
    "eva": "Deutsche Bahn station identifier",
    "train_number": "Timetables XML tl.n",
    "line_number": "Timetables XML ar.l or dp.l",
    "final_destination_station": "Monthly processing logic",
    "delay_in_min": "Derived from planned/effective event times",
    "time": "Monthly arrival/departure selection logic",
    "is_canceled": "Arrival/departure cancellation fields",
    "train_type": "Timetables XML tl.c",
    "train_line_ride_id": "Source processing logic",
    "train_line_station_num": "Parsed from source stop id",
    "arrival_planned_time": "Timetables XML ar.pt",
    "arrival_change_time": "Timetables XML ar.ct with ar.pt fallback",
    "departure_planned_time": "Timetables XML dp.pt",
    "departure_change_time": "Timetables XML dp.ct with dp.pt fallback",
    "id": "Source processing logic",
}

column_quality_notes = {
    "station_name": (
        "May be null when EVA is absent from the lookup; "
        "compare with xml_station_name."
    ),
    "xml_station_name": (
        "May use a different spelling from station_name."
    ),
    "eva": (
        "Keep as string to preserve leading zeros; "
        "a DB-to-VBB crosswalk needs separate validation."
    ),
    "train_number": (
        "Not globally unique across dates or service types."
    ),
    "line_number": (
        "Non-unique and often null for some long-distance services."
    ),
    "final_destination_station": (
        "Standardize through a mapping table, not destructive cleaning."
    ),
    "delay_in_min": (
        "Profile negative values, extremes, and cancellation behavior."
    ),
    "time": (
        "Business timezone interpretation still requires confirmation."
    ),
    "is_canceled": (
        "Confirm behavior for partially canceled or incomplete stops."
    ),
    "train_type": (
        "Profile unexpected codes before defining a normalization map."
    ),
    "train_line_ride_id": (
        "Expected to repeat across station observations."
    ),
    "train_line_station_num": (
        "Check duplicates, gaps, and resets within each ride."
    ),
    "arrival_planned_time": (
        "May be null at stops without an arrival event."
    ),
    "arrival_change_time": (
        "Fallback to planned time does not prove an on-time arrival."
    ),
    "departure_planned_time": (
        "May be null at stops without a departure event."
    ),
    "departure_change_time": (
        "Fallback to planned time does not prove an on-time departure."
    ),
    "id": (
        "Candidate primary key tested in both validation months."
    ),
}

## 8. Generate the combined technical and business column catalog

This merges:

1. the schema Spark actually read;
2. business descriptions and formats;
3. field order, type, and source nullability;
4. semantic roles, derivations, and quality notes.

Detailed null percentages and distributions remain deferred to the profiling
notebook.

### Rendering approach

The small 17-row catalog is rendered with standard-library HTML. Installing
pandas inside the notebook was removed because runtime package installation
reduces reproducibility and can create environment conflicts.

In [52]:
catalog_rows = []

for position, field in enumerate(
    historical_df_072026_raw.schema.fields,
    start=1,
):
    metadata = column_metadata.get(
        field.name,
        {
            "description": "Not documented yet.",
            "unit_or_format": "unknown",
            "verification_status": "to_verify",
        },
    )

    catalog_rows.append(
        (
            position,
            field.name,
            field.dataType.simpleString(),
            field.nullable,
            column_roles.get(field.name, "unknown"),
            metadata["description"],
            metadata["unit_or_format"],
            column_sources.get(field.name, "unknown"),
            column_quality_notes.get(
                field.name,
                "Not reviewed",
            ),
            metadata["verification_status"],
        )
    )

column_catalog_df = spark.createDataFrame(
    catalog_rows,
    (
        "ordinal_position int, "
        "column_name string, "
        "spark_data_type string, "
        "nullable boolean, "
        "business_role string, "
        "description string, "
        "unit_or_format string, "
        "source_or_derivation string, "
        "data_quality_notes string, "
        "verification_status string"
    ),
)

In [53]:

column_catalog_pd = (
    column_catalog_df
    .orderBy("ordinal_position")
    .drop("ordinal_position")
    .toPandas()
)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

styled_catalog = (
    column_catalog_pd.style
    .hide(axis="index")
    .set_properties(
        subset=["description", "verification_status"],
        **{
            "white-space": "normal",
            "text-align": "left",
            "min-width": "280px",
            "vertical-align": "top",
        }
    )
    .set_properties(
        **{
            "white-space": "normal",
            "vertical-align": "top",
        }
    )
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("text-align", "left"),
                    ("vertical-align", "top"),
                ],
            }
        ]
    )
)

display(styled_catalog)

column_name,spark_data_type,nullable,business_role,description,unit_or_format,source_or_derivation,data_quality_notes,verification_status
station_name,string,True,dimension,Station name resolved from the EVA-to-station-name mapping used during monthly processing. It may be null when the EVA number is not present in that mapping.,"string, nullable",EVA-to-station-name lookup,May be null when EVA is absent from the lookup; compare with xml_station_name.,source_documented
xml_station_name,string,True,dimension,Station name taken from the root `station` attribute of the Deutsche Bahn Timetables XML response.,string,Timetables XML station attribute,May use a different spelling from station_name.,source_documented
eva,string,True,identifier,EVA station number: the Deutsche Bahn station identifier used in Timetables API requests.,string identifier,Deutsche Bahn station identifier,Keep as string to preserve leading zeros; a DB-to-VBB crosswalk needs separate validation.,source_documented
train_number,string,True,identifier,"Raw train number (`tl.n`, Zugnummer) identifying a specific train run, for example `123` or `12603`. Combine with `train_type` for labels such as `ICE 123`.",string,Timetables XML tl.n,Not globally unique across dates or service types.,source_documented
line_number,string,True,dimension,"Raw route or line number from `ar.l` or `dp.l`. It is non-unique across runs and is commonly null for long-distance trains such as ICE, IC, and EC.","string, nullable",Timetables XML ar.l or dp.l,Non-unique and often null for some long-distance services.,source_documented
final_destination_station,string,True,dimension,Final destination of the train.,string,Monthly processing logic,"Standardize through a mapping table, not destructive cleaning.",source_documented
delay_in_min,int,True,measure,Delay in minutes.,integer minutes,Derived from planned/effective event times,"Profile negative values, extremes, and cancellation behavior.",source_documented
time,bigint,True,event_time,Actual arrival or departure time.,"raw long in nanoseconds; decoded as Spark timestamp, business timezone to verify",Monthly arrival/departure selection logic,Business timezone interpretation still requires confirmation.,source_documented
is_canceled,boolean,True,flag,Indicates whether the train stop was canceled. The value is derived from the presence of arrival or departure cancellation timestamps (`ar.clt` or `dp.clt`).,boolean,Arrival/departure cancellation fields,Confirm behavior for partially canceled or incomplete stops.,source_documented
train_type,string,True,dimension,"Raw train category from `tl.c`, for example ICE, IC, RE, RB, or S.",string,Timetables XML tl.c,Profile unexpected codes before defining a normalization map.,source_documented


## 9. Check catalog coverage

A catalog should not silently leave physical columns undocumented.

This check reports:

- physical fields missing from the metadata dictionary;
- metadata entries that do not exist in the loaded schema;
- overall catalog coverage.

In [30]:
physical_columns = set(historical_df_072026_raw.columns)
documented_columns = set(column_metadata.keys())

missing_documentation = sorted(
    physical_columns - documented_columns
)

metadata_without_column = sorted(
    documented_columns - physical_columns
)

print(
    "Physical columns without documentation:",
    missing_documentation,
)

print(
    "Metadata entries without a physical column:",
    metadata_without_column,
)

catalog_coverage_pct = (
    100.0
    * len(physical_columns & documented_columns)
    / len(physical_columns)
)

print(
    f"Catalog coverage: {catalog_coverage_pct:.1f}%"
)

Physical columns without documentation: []
Metadata entries without a physical column: []
Catalog coverage: 100.0%


## 11. Export the data catalog

The notebook exports five structured tables in both Parquet and CSV format:

- `dataset_catalog`;
- `column_catalog`;
- `sample_summary`;
- `schema_comparison`;
- `key_assessment`.

It also writes JSON versions of the dataset and column metadata so nested
values and evidence fields are preserved without flattening.

In [31]:
CATALOG_BASE_PATH.mkdir(
    parents=True,
    exist_ok=True,
)

catalog_outputs = {
    "dataset_catalog": dataset_catalog_df,
    "column_catalog": column_catalog_df,
    "sample_summary": sample_summary_df,
    "schema_comparison": schema_comparison_df,
    "key_assessment": key_assessment_df,
}

for output_name, output_df in catalog_outputs.items():
    parquet_path = str(
        CATALOG_BASE_PATH
        / f"{output_name}_parquet"
    )

    csv_path = str(
        CATALOG_BASE_PATH
        / f"{output_name}_csv"
    )

    (
        output_df.write
        .mode("overwrite")
        .parquet(parquet_path)
    )

    (
        output_df.coalesce(1)
        .write
        .mode("overwrite")
        .option("header", True)
        .csv(csv_path)
    )

    print(f"Saved {output_name}:")
    print("  Parquet:", parquet_path)
    print("  CSV:", csv_path)

dataset_catalog_json_path = (
    CATALOG_BASE_PATH / "dataset_catalog.json"
)

column_metadata_json_path = (
    CATALOG_BASE_PATH / "column_metadata.json"
)

dataset_catalog_json_path.write_text(
    json.dumps(
        {
            "catalog": dataset_catalog,
            "evidence": dataset_catalog_evidence,
        },
        indent=2,
        ensure_ascii=False,
        default=str,
    ),
    encoding="utf-8",
)

column_metadata_json_path.write_text(
    json.dumps(
        {
            "column_metadata": column_metadata,
            "column_roles": column_roles,
            "column_sources": column_sources,
            "column_quality_notes": column_quality_notes,
        },
        indent=2,
        ensure_ascii=False,
        default=str,
    ),
    encoding="utf-8",
)

print("Saved JSON:", dataset_catalog_json_path)
print("Saved JSON:", column_metadata_json_path)

Saved dataset_catalog:
  Parquet: /opt/spark/work-dir/artifacts/catalog/deutsche_bahn_historical/dataset_catalog_parquet
  CSV: /opt/spark/work-dir/artifacts/catalog/deutsche_bahn_historical/dataset_catalog_csv
Saved column_catalog:
  Parquet: /opt/spark/work-dir/artifacts/catalog/deutsche_bahn_historical/column_catalog_parquet
  CSV: /opt/spark/work-dir/artifacts/catalog/deutsche_bahn_historical/column_catalog_csv
Saved sample_summary:
  Parquet: /opt/spark/work-dir/artifacts/catalog/deutsche_bahn_historical/sample_summary_parquet
  CSV: /opt/spark/work-dir/artifacts/catalog/deutsche_bahn_historical/sample_summary_csv
Saved schema_comparison:
  Parquet: /opt/spark/work-dir/artifacts/catalog/deutsche_bahn_historical/schema_comparison_parquet
  CSV: /opt/spark/work-dir/artifacts/catalog/deutsche_bahn_historical/schema_comparison_csv
Saved key_assessment:
  Parquet: /opt/spark/work-dir/artifacts/catalog/deutsche_bahn_historical/key_assessment_parquet
  CSV: /opt/spark/work-dir/artifacts/

## Conclusion

Passing two monthly samples is an **initial validation**, not proof for the
entire historical archive. 